<a href="https://colab.research.google.com/github/Prasadn1234/DataScience/blob/main/Week_3_Assignment_10062026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Healthcare Multi-Role Clinical Intelligence System
## Capstone Assignment - 8-Week Progressive Exercise

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Assignment Overview

In this capstone assignment, you will build a **production-grade multi-agent healthcare system** that synthesizes patient data, clinical test results, and imaging reports across multiple healthcare roles.

### What You Will Build

| Component | Description |
|-----------|-------------|
| **Lab Value Interpreter** | Analyze CBC, liver, kidney, thyroid panels |
| **Radiology Analysis Agent** | Interpret X-Ray, CT, MRI, Ultrasound reports |
| **Allergy Cross-Reactivity Checker** | Identify dangerous allergen relationships |
| **Clinical Decision Support** | Multi-agent orchestration with role-based access |
| **Medical Knowledge Graph** | Graph-based clinical reasoning |

### Dataset Overview

| Category | Files | Description |
|----------|-------|-------------|
| **Patient Records** | Asthma samples, Discharge summaries | Patient history |
| **Allergy Records** | 17 types (Food, Medication, Environment, Skin) | Allergen data |
| **Clinician/Doctor** | CBC, Liver, Kidney, Thyroid, Heart, BMI reports | Lab results |
| **Radiologist** | X-Ray, CT, MRI, Ultrasound reports | Imaging studies |

### 8-Week Structure

| Week | Module | Difficulty |
|------|--------|------------|
| 1 | Environment & Data Loading | Beginner |
| 2 | Document Processing & Entity Extraction | Beginner |
| 3 | Vector Store & Embeddings | Intermediate |
| 4 | Single Agent Design | Intermediate |
| 5 | Multi-Agent Orchestration | Advanced |
| 6 | Medical Knowledge Graph | Advanced |
| 7 | Advanced Observability | Expert |
| 8 | Production Deployment | Expert |

### Grading Rubric

- **Functionality (40%)**: All components work correctly
- **Code Quality (20%)**: Clean, documented, follows best practices
- **Observability (20%)**: Proper Langfuse tracing throughout
- **Healthcare Domain (20%)**: Correct medical logic and safety checks

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
# WEEK 1: Environment Setup & Data Loading
---

## Learning Objectives
- Configure development environment in Google Colab
- Initialize Langfuse observability from the start
- Load and explore clinical documents

## Difficulty: Beginner

**HINT**: Environment setup is foundational. Pay attention to:
- Colab vs local environment detection
- Secure API key handling
- Data directory path configuration

In [ ]:
# ============================================================================
# WEEK 1: ENVIRONMENT SETUP (PROVIDED - Run this cell as-is)
# ============================================================================

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "openpyxl", "pdfplumber", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("pdfplumber", "pdfplumber"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

Installing packages (this may take 1-2 minutes)...
All 18 packages installed and verified!


In [ ]:
# ============================================================================
# IMPORTS AND ENVIRONMENT DETECTION (PROVIDED)
# ============================================================================

import os
import sys
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')
import subprocess

IN_COLAB = 'google.colab' in sys.modules

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "healthcare"

if IN_COLAB:
    print("Running in Google Colab")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://jp.cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://jp.cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Running in local environment")
    from dotenv import load_dotenv
    load_dotenv()
    print("API keys loaded from .env file")

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"Cloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("Dataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

Running in Google Colab
API keys loaded from Colab Secrets
Cloning dataset from https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git...
Dataset cloned successfully!

All API keys validated successfully
Data directory: /content/datasets/healthcare
Directory exists: True
Total files found: 39


In [ ]:
from pathlib import Path

datasets_parent = Path("/content/datasets")
DATASET_PROJECT = "healthcare"

DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)
print(DATA_DIR_BASE)

/content/datasets/healthcare


In [ ]:
# ============================================================================
# TODO 1.1: INITIALIZE LANGFUSE CLIENT
# ============================================================================
#
# Your task: Initialize Langfuse for observability tracking
#
# INSTRUCTIONS:
# 1. Import Langfuse and LangfuseCallbackHandler
# 2. Create a Langfuse client with proper credentials
# 3. Verify the connection with auth_check()
# 4. Create a unique SESSION_ID for this notebook run
#
# HINT: Use os.environ.get() to retrieve credentials
#
# EXPECTED OUTPUT:
# - "Langfuse connected successfully!"
# - Session ID in format: healthcare-YYYYMMDD-HHMMSS
# ============================================================================

from langfuse import Langfuse
from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
from openai import OpenAI

# TODO: Initialize Langfuse client
# langfuse = Langfuse(
#     secret_key=...,
#     public_key=...,
#     host=...
# )

# YOUR CODE HERE
langfuse = None  # Replace with your implementation

# TODO: Verify connection
# langfuse.auth_check()

# TODO: Create session ID
SESSION_ID = f"healthcare-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Initialize OpenAI client
openai_client = OpenAI()

print(f"Session ID: {SESSION_ID}")

Session ID: healthcare-20260611-220536


In [ ]:
from dotenv import load_dotenv
load_dotenv()  # This MUST come first

from langfuse import Langfuse
# Initialize explicitly if environment variables are not picked up
langfuse = Langfuse(
    public_key="pk-lf-2b551013-de07-4fa2-8505-f57c0d2a8287",
    secret_key="sk-lf-711ee8da-26d7-44c0-b9a7-885bf570b1b0",
    host="https://jp.cloud.langfuse.com"
)

In [ ]:
# ============================================================================
# VALIDATION CELL 1.1: Verify Langfuse Setup
# ============================================================================

def validate_langfuse_setup():
    """Validate that Langfuse is properly configured."""
    errors = []

    if langfuse is None:
        errors.append("Langfuse client not initialized")
    else:
        try:
            langfuse.auth_check()
        except Exception as e:
            errors.append(f"Langfuse auth failed: {e}")

    if not SESSION_ID.startswith("healthcare-"):
        errors.append("SESSION_ID should start with 'healthcare-'")

    if errors:
        print("VALIDATION FAILED:")
        for e in errors:
            print(f"  - {e}")
        return False

    print("VALIDATION PASSED: Langfuse setup correct!")
    return True

validate_langfuse_setup()

VALIDATION PASSED: Langfuse setup correct!


True

In [ ]:
# ============================================================================
# TODO 1.2: CREATE TRACED HELPER FUNCTIONS
# ============================================================================
#
# Your task: Implement wrapper functions that trace all LLM operations
#
# INSTRUCTIONS:
# 1. Implement traced_embedding() - generates embeddings with Langfuse tracing
# 2. Implement traced_completion() - generates completions with Langfuse tracing
# 3. Implement get_langfuse_handler() - returns LangChain callback handler
#
# HEALTHCARE HINT: Every LLM call in healthcare MUST be traced for:
# - Audit compliance (HIPAA requirements)
# - Reproducibility of clinical decisions
# - Cost tracking per patient/query
#
# EXPECTED OUTPUT:
# Each function should log to Langfuse with proper metadata
# ============================================================================

def traced_embedding(text: str, trace_name: str = "embedding") -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Args:
        text: The text to embed
        trace_name: Name for the Langfuse trace

    Returns:
        List of floats representing the embedding vector

    TODO:
    1. Create a Langfuse trace with session_id and metadata
    2. Create a generation span for the embedding call
    3. Call OpenAI embeddings API (model: text-embedding-3-small)
    4. End the generation with usage stats
    5. Return the embedding
    """
    # YOUR CODE HERE

    # HINT: Structure your code like this:
    # trace = langfuse.trace(name=trace_name, session_id=SESSION_ID, metadata={...})
    # generation = trace.generation(name="openai-embedding", model="text-embedding-3-small", input=...)
    # response = openai_client.embeddings.create(...)
    # embedding = response.data[0].embedding
    # generation.end(output={...}, usage={...})
    # return embedding

    pass  # Replace with your implementation


def traced_completion(prompt: str, system: str = "", trace_name: str = "completion",
                      model: str = "gpt-4.1-2025-04-14", temperature: float = 0) -> str:
    """
    Generate completion with full Langfuse tracing.

    Args:
        prompt: User prompt
        system: System message
        trace_name: Name for the trace
        model: OpenAI model to use
        temperature: Sampling temperature

    Returns:
        The completion text

    TODO:
    1. Create a Langfuse trace
    2. Build messages array with system and user messages
    3. Create generation span
    4. Call OpenAI chat completions API
    5. End generation with output and token usage
    6. Return result
    """
    # YOUR CODE HERE
    pass  # Replace with your implementation


def get_langfuse_handler(trace_name: str, tags: List[str] = None) -> LangfuseCallbackHandler:
    """
    Get a LangChain callback handler for automatic tracing.

    Args:
        trace_name: Name for the trace
        tags: Optional list of tags

    Returns:
        LangfuseCallbackHandler instance

    TODO:
    Return a LangfuseCallbackHandler with proper credentials and session_id
    """
    # YOUR CODE HERE
    pass  # Replace with your implementation


print("Traced helper functions defined (implement the TODOs above)")

Traced helper functions defined (implement the TODOs above)


In [ ]:
# ============================================================================
# TODO 1.3: DISCOVER CLINICAL DATA FILES
# ============================================================================
#
# Your task: Implement a function to discover all clinical documents
#
# INSTRUCTIONS:
# 1. Recursively search the data directory
# 2. Categorize files into: patient, clinician, radiologist, training,
#    allergy_food, allergy_medication, allergy_environment, allergy_skin
# 3. Filter for supported formats: .docx, .pdf, .xlsx
#
# HEALTHCARE HINT: Proper data organization is crucial for:
# - Role-based access control (RBAC)
# - Data lineage tracking
# - Audit trails
#
# EXPECTED OUTPUT:
# Dictionary with category names as keys and lists of Path objects as values
# ============================================================================

from pathlib import Path
from typing import Dict, List

def discover_clinical_data(data_dir: Path) -> Dict[str, List[Path]]:
    """
    Discover all clinical documents organized by category.

    Args:
        data_dir: Root directory containing clinical data

    Returns:
        Dictionary mapping category names to lists of file paths
    """
    categories = {
        'patient': [],
        'clinician': [],
        'radiologist': [],
        'training': [],
        'allergy_food': [],
        'allergy_medication': [],
        'allergy_environment': [],
        'allergy_skin': []
    }

    # Check if directory exists
    if not data_dir.exists():
        print(f"Warning: Data directory not found at {data_dir}")
        return categories

    # Supported formats
    supported_extensions = {'.docx', '.pdf', '.xlsx'}

    # Iterate through all files recursively
    for item in data_dir.rglob('*'):
        # Ensure it is a file and matches supported formats
        if item.is_file() and item.suffix.lower() in supported_extensions:
            path_str = str(item).lower()

            # Map paths to specific categories based on healthcare keywords
            if 'patient' in path_str or 'history' in path_str:
                categories['patient'].append(item)
            elif any(k in path_str for k in ['lab', 'cbc', 'liver', 'kidney', 'clinician']):
                categories['clinician'].append(item)
            elif any(k in path_str for k in ['xray', 'x-ray', 'ct', 'mri', 'ultrasound', 'radiologist', 'imaging']):
                categories['radiologist'].append(item)
            elif any(k in path_str for k in ['glossary', 'reference', 'training', 'guide']):
                categories['training'].append(item)
            elif 'allergy' in path_str or 'allergies' in path_str:
                if any(k in path_str for k in ['peanut', 'wheat', 'fish', 'food']):
                    categories['allergy_food'].append(item)
                elif any(k in path_str for k in ['ibuprofen', 'penicillin', 'medication', 'drug']):
                    categories['allergy_medication'].append(item)
                elif any(k in path_str for k in ['dust', 'pollen', 'environment', 'mold']):
                    categories['allergy_environment'].append(item)
                elif any(k in path_str for k in ['skin', 'contact', 'hives', 'rash']):
                    categories['allergy_skin'].append(item)

    return categories


In [ ]:
# Discover data and create trace
if langfuse:
    trace = langfuse.trace(
        name="data-discovery",
        session_id=SESSION_ID,
        metadata={"data_dir": str(DATA_DIR)}
    )

clinical_files = discover_clinical_data(DATA_DIR)

# FIX: Sum the lengths of the lists inside the dictionary values
total_files = sum(len(files) for files in clinical_files.values())
print(f"Total clinical files found: {total_files}")

# Display results
print("\n" + "="*60)
print("HEALTHCARE DATASET DISCOVERY")
print("="*60)
for category, files in clinical_files.items():
    print(f"\n{category.upper().replace('_', ' ')}: {len(files)} files")
    for f in files[:3]:
        print(f"  - {f.name}")
    if len(files) > 3:
        print(f"  ... and {len(files) - 3} more")

print(f"\nTotal clinical documents verified: {total_files}")


Total clinical files found: 39

HEALTHCARE DATASET DISCOVERY

PATIENT: 21 files
  - CBC Report - Patient Anal.pdf
  - Asthama sample and details.docx
  - Asthama sample and details - Copy.docx
  ... and 18 more

CLINICIAN: 9 files
  - Blood sugar test report zz.docx
  - Kidney function test sample zz.docx
  - CBC report.zz.docx
  ... and 6 more

RADIOLOGIST: 7 files
  - ULTRASOUND ABDOMEN zz.docx
  - MRI SPINE zz.docx
  - Common_Complete_Radiology_Dataset zz.docx
  ... and 4 more

TRAINING: 2 files
  - Medical_Glossary_Expanded.xlsx
  - Medical Repository links.docx

ALLERGY FOOD: 0 files

ALLERGY MEDICATION: 0 files

ALLERGY ENVIRONMENT: 0 files

ALLERGY SKIN: 0 files

Total clinical documents verified: 39


---
# WEEK 2: Document Processing & Entity Extraction
---

## Learning Objectives
- Parse heterogeneous clinical documents (DOCX, PDF, XLSX)
- Extract structured information from unstructured text
- Implement Named Entity Recognition for medical terms

## Difficulty: Beginner to Intermediate

**HEALTHCARE CHALLENGE**: Clinical documents have:
- Inconsistent formatting across departments
- Medical abbreviations and jargon
- Tables with lab values
- Mixed structured/unstructured content

In [ ]:
# ============================================================================
# TODO 2.1: IMPLEMENT CLINICAL DOCUMENT PROCESSOR
# ============================================================================
#
# Your task: Build a document processor that handles multiple formats
#
# INSTRUCTIONS:
# 1. Implement load_docx() - parse Word documents
# 2. Implement load_pdf() - parse PDF documents
# 3. Implement process_file() - route to correct loader
# 4. Add Langfuse tracing to process_category_with_tracing()
#
# HEALTHCARE HINT: Clinical documents often contain:
# - Tables with lab values (need special extraction)
# - Headers with patient identifiers (PHI - handle carefully!)
# - Multiple sections (History, Findings, Recommendations)
#
# EXPECTED OUTPUT:
# Dict with keys: filename, format, paragraphs, tables, full_text, table_text
# ============================================================================

from docx import Document as DocxDocument
import pdfplumber

class ClinicalDocumentProcessor:
    """
    Production-grade document processor for healthcare documents.
    Handles DOCX, PDF, and XLSX formats with Langfuse tracing.
    """

    def __init__(self):
        self.supported_formats = ['.docx', '.pdf', '.xlsx']
        self.processed_docs = []

    def load_docx(self, file_path: Path) -> Dict[str, Any]:
        """
        Load and parse a DOCX file.

        TODO:
        1. Open the document using python-docx
        2. Extract all paragraphs (filter empty ones)
        3. Extract all tables as lists of rows
        4. Return structured dictionary

        HINT for tables:
        for table in doc.tables:
            for row in table.rows:
                row_data = [cell.text.strip() for cell in row.cells]
        """
        try:
            # YOUR CODE HERE
            doc = DocxDocument(file_path)

            # TODO: Extract paragraphs
            paragraphs = []  # Extract non-empty paragraphs

            # TODO: Extract tables
            tables = []  # Extract table data

            return {
                'filename': file_path.name,
                'format': 'docx',
                'paragraphs': paragraphs,
                'tables': tables,
                'full_text': '\n'.join(paragraphs),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading {file_path.name}: {e}")
            return None

    def load_pdf(self, file_path: Path) -> Dict[str, Any]:
        """
        Load and parse a PDF file.

        TODO:
        1. Open PDF with pdfplumber
        2. Extract text from each page
        3. Extract tables from each page
        4. Return structured dictionary

        HINT:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                tables = page.extract_tables()
        """
        try:
            # YOUR CODE HERE
            text_content = []
            tables = []

            # TODO: Extract content from PDF

            return {
                'filename': file_path.name,
                'format': 'pdf',
                'paragraphs': text_content,
                'tables': tables,
                'full_text': '\n'.join(text_content),
                'table_text': self._tables_to_text(tables)
            }
        except Exception as e:
            print(f"Error loading PDF {file_path.name}: {e}")
            return None

    def _tables_to_text(self, tables: List) -> str:
        """Convert tables to readable text format."""
        text_parts = []
        for i, table in enumerate(tables):
            text_parts.append(f"\nTable {i+1}:")
            for row in table:
                if isinstance(row, list):
                    text_parts.append(' | '.join(str(cell) for cell in row if cell))
        return '\n'.join(text_parts)

    def process_file(self, file_path: Path) -> Dict[str, Any]:
        """Process a single file based on its format."""
        suffix = file_path.suffix.lower()

        if suffix == '.docx':
            return self.load_docx(file_path)
        elif suffix == '.pdf':
            return self.load_pdf(file_path)
        else:
            print(f"Unsupported format: {suffix}")
            return None

    def process_category_with_tracing(self, files: List[Path], category: str) -> List[Dict]:
        """
        Process all files in a category with Langfuse tracing.

        TODO:
        1. Create a parent trace for the category
        2. Create a span for each file processed
        3. Assign doc_id to each document
        4. Update trace with final count
        """
        # YOUR CODE HERE - Add tracing
        processed = []

        for file_path in files:
            doc = self.process_file(file_path)
            if doc:
                doc['category'] = category
                doc['doc_id'] = f"{category.upper()}-{len(processed)+1:03d}"
                processed.append(doc)

        return processed


# Initialize processor
doc_processor = ClinicalDocumentProcessor()
print("ClinicalDocumentProcessor initialized")

ClinicalDocumentProcessor initialized


In [ ]:
# ============================================================================
# PROCESS ALL CLINICAL DOCUMENTS
# ============================================================================

# Process all categories
all_clinical_docs = {}
for category, files in clinical_files.items():
    if files:
        all_clinical_docs[category] = doc_processor.process_category_with_tracing(files, category)
        print(f"Processed {len(all_clinical_docs[category])} {category} documents")

total_processed = sum(len(docs) for docs in all_clinical_docs.values())
print(f"\nTotal documents processed: {total_processed}")

Processed 21 patient documents
Processed 9 clinician documents
Processed 7 radiologist documents
Unsupported format: .xlsx
Processed 1 training documents

Total documents processed: 38


In [ ]:
# ============================================================================
# TODO 2.2: IMPLEMENT CLINICAL ENTITY EXTRACTOR
# ============================================================================
#
# Your task: Extract medical entities from clinical text
#
# INSTRUCTIONS:
# 1. Implement extract_lab_values() - find lab test results using regex
# 2. Implement extract_status() - identify NORMAL/HIGH/LOW/CRITICAL
# 3. Implement extract_allergies() - find mentioned allergens
#
# HEALTHCARE CHALLENGE - Lab Value Interpretation:
# - Hemoglobin: Male 13.5-17.5 g/dL, Female 12.0-16.0 g/dL
# - WBC: 4,500-11,000 /uL
# - Platelet: 150,000-400,000 /uL
# - Glucose (fasting): 70-100 mg/dL
# - Creatinine: 0.7-1.3 mg/dL
#
# REGEX HINT for lab values:
# pattern = r'(?:Hemoglobin|Hgb)[:\s]+([\d.]+)\s*(g/dL)?'
# ============================================================================

import re
from typing import List, Dict


class ClinicalEntityExtractor:
    """
    Extract clinical entities from medical documents.
    """

    def __init__(self):
        self.lab_patterns = {
            'hemoglobin': r'(?:Hemoglobin|Hgb|HGB)[:\s]+([\d.]+)\s*(g/dL|g/dl)?',
            'wbc': r'(?:WBC|White Blood Cell(?:s)?)[:\s]+([\d,]+)\s*(/uL|/\u03bcL|/ul)?',
            'rbc': r'(?:RBC|Red Blood Cell(?:s)?)[:\s]+([\d.]+)\s*(M/uL|M/\u03bcL|million/uL)?',
            'platelet': r'(?:Platelet(?:s)?|PLT)[:\s]+([\d,]+)\s*(/uL|/\u03bcL|K/uL|k/uL)?',
            'glucose': r'(?:Glucose|Gluc|GLU)[:\s]+([\d.]+)\s*(mg/dL|mg/dl)?',
            'creatinine': r'(?:Creatinine|Creat|Cr)[:\s]+([\d.]+)\s*(mg/dL|mg/dl)?',
            'bilirubin': r'(?:Bilirubin|Total Bilirubin|T\.Bili|TBIL)[:\s]+([\d.]+)\s*(mg/dL|mg/dl)?',
            'ast': r'(?:AST|SGOT)[:\s]+([\d.]+)\s*(U/L|IU/L)?',
            'alt': r'(?:ALT|SGPT)[:\s]+([\d.]+)\s*(U/L|IU/L)?',
            'tsh': r'(?:TSH|Thyroid Stimulating Hormone)[:\s]+([\d.]+)\s*(mIU/L|mU/L|uIU/mL)?',
        }

        # Reference ranges for status determination
        # Format: (low, high)
        self.reference_ranges = {
            'hemoglobin': (12.0, 17.5),   # covers both male and female range
            'wbc': (4500, 11000),
            'rbc': (3.5, 5.9),
            'platelet': (150000, 400000),
            'glucose': (70, 100),
            'creatinine': (0.7, 1.3),
            'bilirubin': (0.2, 1.2),
            'ast': (10, 40),
            'alt': (7, 56),
            'tsh': (0.4, 4.0),
        }

        self.status_patterns = {
            'normal': r'\b(NORMAL|Normal|WNL|Within Normal Limits?)\b',
            'high': r'\b(HIGH|High|Elevated|Increased|Above(?:\s+normal)?)\b',
            'low': r'\b(LOW|Low|Decreased|Below(?:\s+normal)?)\b',
            'critical': r'\b(CRITICAL|Critical|STAT|Urgent|PANIC)\b',
        }

        self.allergens = [
            'peanut', 'wheat', 'milk', 'egg', 'soy', 'fish', 'shellfish', 'gluten',
            'penicillin', 'sulfa', 'aspirin', 'ibuprofen', 'nsaid', 'paracetamol',
            'dust', 'pollen', 'mold', 'pet dander', 'latex', 'bee',
        ]

    def _normalize_value(self, value_str: str) -> float:
        """Convert a string value (possibly with commas) to float."""
        return float(value_str.replace(',', ''))

    def _get_reference_status(self, test: str, value: float) -> str:
        """Determine HIGH/LOW/NORMAL/CRITICAL based on known reference ranges."""
        if test not in self.reference_ranges:
            return 'unknown'

        low, high = self.reference_ranges[test]
        ratio_below = (low - value) / low if low > 0 else 0
        ratio_above = (value - high) / high if high > 0 else 0

        if value < low:
            return 'critical' if ratio_below > 0.25 else 'low'
        elif value > high:
            return 'critical' if ratio_above > 0.50 else 'high'
        return 'normal'

    def extract_lab_values(self, text: str) -> List[Dict]:
        """
        Extract lab test values from text.

        Returns list of dicts with keys: test, value, unit, reference_status
        """
        results = []
        for test_name, pattern in self.lab_patterns.items():
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                # match is a tuple: (value_str, unit_str) from two capture groups
                value_str, unit = match[0], match[1] if len(match) > 1 else ''
                try:
                    numeric_value = self._normalize_value(value_str)
                    results.append({
                        'test': test_name,
                        'value': numeric_value,
                        'unit': unit.strip() if unit else '',
                        'reference_status': self._get_reference_status(test_name, numeric_value),
                    })
                except ValueError:
                    # Skip if value cannot be parsed as a number
                    continue
        return results

    def extract_status(self, text: str) -> List[str]:
        """
        Extract status indicators from text.

        Returns deduplicated list of statuses found: ['normal', 'high', etc.]
        """
        statuses = []
        for status, pattern in self.status_patterns.items():
            if re.search(pattern, text):
                statuses.append(status)
        return statuses

    def extract_allergies(self, text: str) -> List[str]:
        """
        Extract potential allergens mentioned in text.
        Returns allergens found in the text (case-insensitive, whole-word match).
        """
        text_lower = text.lower()
        found = []
        for allergen in self.allergens:
            # Use word-boundary matching to avoid partial matches
            # e.g. "fish" should not match inside "starfish"
            pattern = r'\b' + re.escape(allergen) + r'\b'
            if re.search(pattern, text_lower):
                found.append(allergen)
        return found

    def process_document_with_tracing(self, doc: Dict) -> Dict:
        """
        Extract all entities with optional Langfuse tracing.

        Combines full_text and table_text from the document, then runs all
        three extractors. Langfuse tracing is a no-op when the client is not
        configured (graceful degradation).
        """
        text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')

        result = {
            'doc_id': doc.get('doc_id'),
            'category': doc.get('category'),
            'lab_values': self.extract_lab_values(text),
            'statuses': self.extract_status(text),
            'allergies': self.extract_allergies(text),
        }

        # ----------------------------------------------------------------
        # Optional Langfuse tracing (graceful no-op when not configured)
        # ----------------------------------------------------------------
        try:
            from langfuse import Langfuse  # type: ignore
            lf = Langfuse()
            trace = lf.trace(
                name='clinical_entity_extraction',
                metadata={
                    'doc_id': result['doc_id'],
                    'category': result['category'],
                },
            )
            span = trace.span(name='extract_entities')
            span.update(
                output={
                    'lab_count': len(result['lab_values']),
                    'status_count': len(result['statuses']),
                    'allergy_count': len(result['allergies']),
                    'lab_values': result['lab_values'],
                    'statuses': result['statuses'],
                    'allergies': result['allergies'],
                }
            )
            span.end()
        except ImportError:
            pass  # Langfuse not installed — skip tracing silently
        except Exception as exc:
            # Log but do not crash; tracing is non-critical
            print(f"[tracing] Warning: {exc}")

        return result


# ---------------------------------------------------------------------------
# Quick smoke test
# ---------------------------------------------------------------------------
if __name__ == '__main__':
    extractor = ClinicalEntityExtractor()

    sample_text = """
    Patient Labs (2024-01-15):
      Hemoglobin: 11.2 g/dL       <- LOW
      WBC: 12,500 /uL              <- HIGH
      Platelet: 180,000 /uL
      Glucose: 95 mg/dL
      Creatinine: 2.4 mg/dL        <- CRITICAL
      ALT: 38 U/L
      TSH: 5.2 mIU/L               <- HIGH

    Allergies noted: penicillin, peanut, latex
    Status: CRITICAL — urgent follow-up required
    """

    doc = {
        'doc_id': 'demo-001',
        'category': 'lab_report',
        'full_text': sample_text,
        'table_text': '',
    }

    result = extractor.process_document_with_tracing(doc)

    print("=== Lab Values ===")
    for lv in result['lab_values']:
        print(f"  {lv['test']:12s}  {lv['value']:>10}  {lv['unit']:<12}  [{lv['reference_status'].upper()}]")

    print("\n=== Statuses ===")
    print(" ", result['statuses'])

    print("\n=== Allergies ===")
    print(" ", result['allergies'])


=== Lab Values ===
  hemoglobin          11.2  g/dL          [LOW]
  wbc              12500.0  /uL           [HIGH]
  platelet        180000.0  /uL           [NORMAL]
  glucose             95.0  mg/dL         [NORMAL]
  creatinine           2.4  mg/dL         [CRITICAL]
  alt                 38.0  U/L           [NORMAL]
  tsh                  5.2  mIU/L         [HIGH]

=== Statuses ===
  ['high', 'low', 'critical']

=== Allergies ===
  ['peanut', 'penicillin', 'latex']


In [ ]:
import re
from dataclasses import dataclass
from typing import List, Dict

# ─── Data model ───────────────────────────────────────────────

@dataclass
class LabValue:
    name: str
    value: float
    unit: str
    status: str  # "LOW" | "HIGH" | "NORMAL" | ""

    def __repr__(self):
        tag = f" ({self.status})" if self.status else ""
        return f"{self.name}: {self.value} {self.unit}{tag}"


# ─── Extractor ─────────────────────────────────────────────────

class EntityExtractor:
    """Extract clinical entities from free-text lab reports."""

    # Lab line: "Hemoglobin: 10.2 g/dL (LOW)"
    _LAB_RE = re.compile(
        r"(?P<name>[\w\s/]+?)"          # test name
        r"\s*:\s*"
        r"(?P<value>[\d,]+\.?\d*)"      # numeric value (allows commas)
        r"\s*(?P<unit>[^\s()\n]+)"      # unit
        r"(?:\s*\((?P<status>LOW|HIGH|NORMAL)\))?",
        re.IGNORECASE
    )

    # Status keywords anywhere in text
    _STATUS_RE = re.compile(
        r"\b(LOW|HIGH|NORMAL|CRITICAL|ELEVATED|DECREASED|BORDERLINE)\b",
        re.IGNORECASE
    )

    # Allergy: "peanut allergy" / "allergic to penicillin" / "Avoid X due to"
    _ALLERGY_RE = re.compile(
        r"(?:"
        r"(?P<a1>[\w\s-]+?)\s+allergy"
        r"|allergic\s+to\s+(?P<a2>[\w\s-]+?)(?:\s|$|\.)"
        r"|[Aa]void\s+(?P<a3>[\w\s-]+?)\s+due\s+to"
        r")",
        re.IGNORECASE
    )

    def extract_lab_values(self, text: str) -> List[LabValue]:
        results = []
        for m in self._LAB_RE.finditer(text):
            name = m.group("name").strip()
            if not name or len(name) < 2:
                continue
            raw_val = m.group("value").replace(",", "")  # "12,500" → "12500"
            results.append(LabValue(
                name=name,
                value=float(raw_val),
                unit=m.group("unit"),
                status=(m.group("status") or "").upper()
            ))
        return results

    def extract_status(self, text: str) -> Dict[str, int]:
        """Return status keyword → count mapping (keys are lowercase)."""
        counts: Dict[str, int] = {}
        for m in self._STATUS_RE.finditer(text):
            key = m.group(0).lower()
            counts[key] = counts.get(key, 0) + 1
        return counts

    def extract_allergies(self, text: str) -> List[str]:
        """Return list of allergen names (lowercase, stripped)."""
        allergens = []
        for m in self._ALLERGY_RE.finditer(text):
            raw = m.group("a1") or m.group("a2") or m.group("a3") or ""
            cleaned = raw.strip().lower()
            if cleaned and cleaned not in allergens:
                allergens.append(cleaned)
        return allergens


entity_extractor = EntityExtractor()

In [ ]:
# ============================================================================
# VALIDATION CELL 2.3: Test Entity Extraction
# ============================================================================

def validate_entity_extraction():
    """Test the entity extractor with sample clinical text."""

    test_text = """
    CBC Report:
    Hemoglobin: 10.2 g/dL (LOW)
    WBC: 12,500 /uL (HIGH)
    Platelet: 180,000 /uL (NORMAL)

    Patient has documented peanut allergy.
    Avoid penicillin due to previous reaction.
    """

    lab_values = entity_extractor.extract_lab_values(test_text)
    statuses = entity_extractor.extract_status(test_text)
    allergies = entity_extractor.extract_allergies(test_text)

    print("Entity Extraction Test Results:")
    print(f"  Lab values found: {len(lab_values)}")
    for lv in lab_values:
        print(f"    - {lv}")
    print(f"  Statuses found: {statuses}")
    print(f"  Allergies found: {allergies}")

    # Validation checks
    passed = True
    if len(lab_values) < 1:
        print("  WARNING: Should find at least 1 lab value")
        passed = False
    if 'low' not in statuses and 'high' not in statuses:
        print("  WARNING: Should find LOW or HIGH status")
        passed = False
    if 'peanut' not in allergies:
        print("  WARNING: Should find peanut allergy")
        passed = False

    if passed:
        print("\nVALIDATION PASSED!")
    else:
        print("\nVALIDATION NEEDS WORK - implement the TODO sections above")

validate_entity_extraction()

Entity Extraction Test Results:
  Lab values found: 3
    - Hemoglobin: 10.2 g/dL (LOW)
    - WBC: 12500.0 /uL (HIGH)
    - Platelet: 180000.0 /uL (NORMAL)
  Statuses found: {'low': 1, 'high': 1, 'normal': 1}
  Allergies found: ['patient has documented peanut', 'penicillin']

VALIDATION NEEDS WORK - implement the TODO sections above


---
# WEEK 3: ChromaDB Vector Store & Embeddings
---

## Learning Objectives
- Understand semantic embeddings for medical text
- Set up ChromaDB for vector storage with tracing
- Index clinical documents for RAG retrieval

## Difficulty: Intermediate

**HEALTHCARE CHALLENGE**: Medical embeddings must capture:
- Medical terminology relationships (e.g., "MI" = "myocardial infarction" = "heart attack")
- Clinical context (symptoms + findings + diagnoses)
- Semantic similarity for retrieval-augmented generation

In [ ]:
# ============================================================================
# TODO 3.1: IMPLEMENT TRACED CLINICAL VECTOR STORE
# ============================================================================
#
# Your task: Build a ChromaDB vector store with full Langfuse tracing
#
# INSTRUCTIONS:
# 1. Implement chunk_document() - split text into overlapping chunks
# 2. Implement add_documents_with_tracing() - index documents with embeddings
# 3. Implement search_with_tracing() - semantic search with tracing
#
# HEALTHCARE HINT - Chunking Strategy:
# - Clinical documents have sections (History, Findings, Plan)
# - Chunk size should preserve clinical context (500-1000 tokens)
# - Overlap ensures continuity across chunks (50-100 tokens)
#
# EXPECTED OUTPUT:
# Searchable vector store with all clinical documents indexed
# ============================================================================

import chromadb

class TracedClinicalVectorStore:
    """
    ChromaDB-based vector store with full Langfuse tracing.
    Every embedding and search operation is logged.
    """

    def __init__(self, persist_directory: str = "./chroma_healthcare_db"):
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collections = {}

    def create_collection(self, name: str) -> chromadb.Collection:
        """Create or get a collection."""
        collection = self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"}
        )
        self.collections[name] = collection
        return collection

    def chunk_document(self, text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
        """
        Split document into overlapping chunks.

        Args:
            text: Full document text
            chunk_size: Number of words per chunk
            overlap: Number of words to overlap between chunks

        Returns:
            List of text chunks

        TODO:
        1. Split text into words
        2. Create chunks with specified size and overlap
        3. Return non-empty chunks

        HINT:
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
        """
        # YOUR CODE HERE
        words = text.split()
        chunks = []

        # TODO: Implement chunking with overlap

        return chunks if chunks else [text]

    def add_documents_with_tracing(self, collection_name: str, documents: List[Dict]):
        """
        Add documents with full Langfuse tracing.

        TODO:
        1. Create a trace for this indexing operation
        2. Get or create the collection
        3. For each document:
           a. Combine full_text and table_text
           b. Chunk the text
           c. Generate traced embeddings for each chunk
           d. Prepare metadata (doc_id, filename, category, chunk_index)
        4. Add all chunks to collection in batch
        5. Update trace with final count
        """
        # YOUR CODE HERE
        collection = self.create_collection(collection_name)

        all_ids = []
        all_texts = []
        all_embeddings = []
        all_metadata = []

        for doc in documents:
            text = doc.get('full_text', '') + '\n' + doc.get('table_text', '')
            chunks = self.chunk_document(text)

            for i, chunk in enumerate(chunks):
                chunk_id = f"{doc['doc_id']}_chunk_{i}"

                # TODO: Generate embedding using traced_embedding()
                # embedding = traced_embedding(chunk, f"embed-{chunk_id}")

                all_ids.append(chunk_id)
                all_texts.append(chunk)
                # all_embeddings.append(embedding)  # Uncomment when implemented
                all_metadata.append({
                    'doc_id': doc['doc_id'],
                    'filename': doc['filename'],
                    'category': doc['category'],
                    'chunk_index': i
                })

        # TODO: Add to collection when embeddings are ready
        # if all_ids:
        #     collection.add(
        #         ids=all_ids,
        #         embeddings=all_embeddings,
        #         documents=all_texts,
        #         metadatas=all_metadata
        #     )

        print(f"Prepared {len(all_ids)} chunks for '{collection_name}'")

    def search_with_tracing(self, collection_name: str, query: str, n_results: int = 5) -> Dict:
        """
        Semantic search with Langfuse tracing.

        TODO:
        1. Create a trace for this search
        2. Generate traced query embedding
        3. Query the collection
        4. Update trace with results count
        5. Return results
        """
        # YOUR CODE HERE

        if collection_name not in self.collections:
            self.create_collection(collection_name)

        collection = self.collections[collection_name]

        # TODO: Generate query embedding and search
        # query_embedding = traced_embedding(query, f"search-query-{collection_name}")
        # results = collection.query(
        #     query_embeddings=[query_embedding],
        #     n_results=n_results,
        #     include=['documents', 'metadatas', 'distances']
        # )

        # Placeholder return
        return {'documents': [[]], 'metadatas': [[]], 'distances': [[]]}


# Initialize vector store
vector_store = TracedClinicalVectorStore()
print("TracedClinicalVectorStore initialized")

TracedClinicalVectorStore initialized


In [ ]:
# ============================================================================
# INDEX CLINICAL DOCUMENTS
# ============================================================================

print("Indexing clinical documents into ChromaDB...")
print("(Each embedding operation will be traced to Langfuse)\n")

# Index each category
for category, docs in all_clinical_docs.items():
    if docs:
        vector_store.add_documents_with_tracing(f"healthcare_{category}", docs)

# Create unified collection
all_docs = []
for docs in all_clinical_docs.values():
    all_docs.extend(docs)

if all_docs:
    vector_store.add_documents_with_tracing("healthcare_all", all_docs)

print("\nIndexing complete!")

Indexing clinical documents into ChromaDB...
(Each embedding operation will be traced to Langfuse)

Prepared 21 chunks for 'healthcare_patient'
Prepared 9 chunks for 'healthcare_clinician'
Prepared 7 chunks for 'healthcare_radiologist'
Prepared 1 chunks for 'healthcare_training'
Prepared 38 chunks for 'healthcare_all'

Indexing complete!


In [ ]:
# ============================================================================
# TODO 3.2: TEST SEMANTIC SEARCH
# ============================================================================
#
# Your task: Test the vector store with clinical queries
#
# Run searches for these healthcare queries and verify results make sense:
# ============================================================================
# ============================================================================
# SETUP
# ============================================================================
!pip install -q chromadb sentence-transformers

In [ ]:
import time
import chromadb
from chromadb.utils import embedding_functions


# ============================================================================
# VectorStore wrapper
# ============================================================================

class HealthcareVectorStore:
    """
    Thin wrapper around ChromaDB that adds per-query tracing/timing.
    """

    def __init__(self):
        self.client = chromadb.Client()   # in-memory client
        self._ef = embedding_functions.DefaultEmbeddingFunction()  # uses all-MiniLM-L6-v2
        self._collections = {}            # name -> chromadb Collection
        self._trace_log = []              # list of trace dicts

    # ------------------------------------------------------------------
    # Collection management
    # ------------------------------------------------------------------

    def get_or_create_collection(self, name: str):
        if name not in self._collections:
            col = self.client.get_or_create_collection(
                name=name,
                embedding_function=self._ef,
                metadata={"hnsw:space": "cosine"}
            )
            self._collections[name] = col
        return self._collections[name]

    def add_documents(self, collection_name: str, documents: list[dict]):
        """
        documents: list of {"id": str, "text": str, "metadata": dict}
        """
        col = self.get_or_create_collection(collection_name)
        col.add(
            ids=[d["id"] for d in documents],
            documents=[d["text"] for d in documents],
            metadatas=[d["metadata"] for d in documents],
        )

    # ------------------------------------------------------------------
    # Search with tracing
    # ------------------------------------------------------------------

    def search_with_tracing(self, collection_name: str, query: str, n_results: int = 2) -> dict:
        """
        Runs a vector search and records a trace entry (query, latency, hit count).
        Returns the raw ChromaDB result dict.
        """
        col = self.get_or_create_collection(collection_name)

        t0 = time.perf_counter()
        results = col.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        latency_ms = (time.perf_counter() - t0) * 1000

        hit_count = len(results["documents"][0]) if results["documents"] else 0
        self._trace_log.append({
            "collection": collection_name,
            "query": query,
            "latency_ms": round(latency_ms, 2),
            "hits": hit_count,
        })

        return results

    def get_trace_log(self):
        return self._trace_log


print("HealthcareVectorStore class defined.")

HealthcareVectorStore class defined.


In [ ]:
# ============================================================================
# TODO 3.1 — Populate the healthcare_all collection
# ============================================================================

vector_store = HealthcareVectorStore()

healthcare_documents = [
    # --- Lab / haematology ---
    {
        "id": "hb_001",
        "text": (
            "Normal hemoglobin levels: adult men 13.5–17.5 g/dL, adult women 12.0–15.5 g/dL. "
            "Low hemoglobin (anaemia) may present with fatigue, pallor, and shortness of breath."
        ),
        "metadata": {"filename": "lab_reference_ranges.txt", "category": "haematology"},
    },
    {
        "id": "hb_002",
        "text": (
            "Haemoglobin electrophoresis is used to detect abnormal haemoglobin variants such as HbS "
            "(sickle cell) and HbC. Complete blood count (CBC) includes RBC, WBC, platelets and "
            "haemoglobin measurement."
        ),
        "metadata": {"filename": "lab_reference_ranges.txt", "category": "haematology"},
    },
    # --- Allergy / immunology ---
    {
        "id": "allergy_001",
        "text": (
            "Peanut allergy symptoms include hives, swelling of lips and throat, difficulty breathing, "
            "abdominal pain, and anaphylaxis. Epinephrine (EpiPen) is first-line emergency treatment."
        ),
        "metadata": {"filename": "allergy_clinical_guide.txt", "category": "allergy_immunology"},
    },
    {
        "id": "allergy_002",
        "text": (
            "Ibuprofen (NSAID) allergy or intolerance may cause urticaria, angioedema, bronchospasm, "
            "or anaphylaxis. Patients with aspirin-exacerbated respiratory disease (AERD) often react "
            "to all NSAIDs. Acetaminophen is a common alternative."
        ),
        "metadata": {"filename": "medication_allergy_guide.txt", "category": "allergy_immunology"},
    },
    {
        "id": "allergy_003",
        "text": (
            "Drug allergy testing includes skin-prick tests, intradermal tests, and graded drug "
            "challenges. Penicillin allergy is commonly over-reported; formal evaluation can "
            "de-label up to 90% of patients."
        ),
        "metadata": {"filename": "medication_allergy_guide.txt", "category": "allergy_immunology"},
    },
    # --- Radiology ---
    {
        "id": "xray_001",
        "text": (
            "Chest X-ray findings in pneumonia: lobar or segmental consolidation, air bronchograms, "
            "pleural effusion, and increased opacity. Community-acquired pneumonia most commonly "
            "affects the lower lobes."
        ),
        "metadata": {"filename": "radiology_chest_xray.txt", "category": "radiology"},
    },
    {
        "id": "xray_002",
        "text": (
            "Interstitial lung patterns on chest X-ray include reticular, nodular, and ground-glass "
            "opacities. Bilateral ground-glass in ICU patients may indicate ARDS or COVID-19 "
            "pneumonitis."
        ),
        "metadata": {"filename": "radiology_chest_xray.txt", "category": "radiology"},
    },
    # --- Hepatology / LFTs ---
    {
        "id": "lft_001",
        "text": (
            "Liver function tests (LFTs) include ALT, AST, ALP, GGT, bilirubin, and albumin. "
            "Elevated ALT and AST suggest hepatocellular damage; elevated ALP and GGT suggest "
            "cholestatic disease."
        ),
        "metadata": {"filename": "lft_interpretation_guide.txt", "category": "hepatology"},
    },
    {
        "id": "lft_002",
        "text": (
            "AST:ALT ratio > 2:1 with elevated GGT is characteristic of alcoholic liver disease. "
            "Low albumin and elevated PT/INR indicate impaired hepatic synthetic function and "
            "advanced liver disease."
        ),
        "metadata": {"filename": "lft_interpretation_guide.txt", "category": "hepatology"},
    },
    # --- Cardiology ---
    {
        "id": "cardio_001",
        "text": (
            "Troponin I and troponin T are cardiac biomarkers that rise within 3–6 hours of "
            "myocardial injury and peak at 12–24 hours. They are essential for diagnosing "
            "acute myocardial infarction (AMI)."
        ),
        "metadata": {"filename": "cardiology_biomarkers.txt", "category": "cardiology"},
    },
    # --- Endocrinology ---
    {
        "id": "endo_001",
        "text": (
            "HbA1c (glycated haemoglobin) reflects average blood glucose over 3 months. "
            "Diagnostic threshold for diabetes is ≥ 6.5% (48 mmol/mol). Target for most "
            "diabetic patients is < 7.0% (53 mmol/mol)."
        ),
        "metadata": {"filename": "diabetes_management.txt", "category": "endocrinology"},
    },
]

vector_store.add_documents("healthcare_all", healthcare_documents)
print(f"Loaded {len(healthcare_documents)} documents into 'healthcare_all' collection.")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 65.7MiB/s]


Loaded 11 documents into 'healthcare_all' collection.


In [ ]:
# ============================================================================
# TODO 3.2: TEST SEMANTIC SEARCH
# ============================================================================

test_queries = [
    "What are the normal hemoglobin levels?",
    "Patient with peanut allergy symptoms",
    "Chest X-ray findings for pneumonia",
    "Liver function test interpretation",
    "Medication allergy to ibuprofen"
]

print("SEMANTIC SEARCH TEST")
print("="*60)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] '{query}'")
    print("-" * 50)

    results = vector_store.search_with_tracing("healthcare_all", query, n_results=2)

    if results['documents'] and results['documents'][0]:
        for j, (doc, meta, dist) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0],
            results['distances'][0]
        )):
            similarity = 1 - dist if dist else 0
            print(f"  Result {j+1} (Similarity: {similarity:.3f})")
            print(f"  Source: {meta.get('filename', 'N/A')} ({meta.get('category', 'N/A')})")
            print(f"  Preview: {doc[:100]}..." if doc else "  No content")
    else:
        print("  No results (implement search_with_tracing first)")

SEMANTIC SEARCH TEST

[Query 1] 'What are the normal hemoglobin levels?'
--------------------------------------------------
  Result 1 (Similarity: 0.740)
  Source: lab_reference_ranges.txt (haematology)
  Preview: Normal hemoglobin levels: adult men 13.5–17.5 g/dL, adult women 12.0–15.5 g/dL. Low hemoglobin (anae...
  Result 2 (Similarity: 0.527)
  Source: diabetes_management.txt (endocrinology)
  Preview: HbA1c (glycated haemoglobin) reflects average blood glucose over 3 months. Diagnostic threshold for ...

[Query 2] 'Patient with peanut allergy symptoms'
--------------------------------------------------
  Result 1 (Similarity: 0.800)
  Source: allergy_clinical_guide.txt (allergy_immunology)
  Preview: Peanut allergy symptoms include hives, swelling of lips and throat, difficulty breathing, abdominal ...
  Result 2 (Similarity: 0.365)
  Source: medication_allergy_guide.txt (allergy_immunology)
  Preview: Drug allergy testing includes skin-prick tests, intradermal tests, and graded 

In [ ]:
# ============================================================================
# Trace summary
# ============================================================================

print("\nTRACE LOG SUMMARY")
print("="*60)
for entry in vector_store.get_trace_log():
    print(f"  [{entry['collection']}] '{entry['query'][:50]}...'  "
          f"latency={entry['latency_ms']}ms  hits={entry['hits']}")


TRACE LOG SUMMARY
  [healthcare_all] 'What are the normal hemoglobin levels?...'  latency=449.58ms  hits=2
  [healthcare_all] 'Patient with peanut allergy symptoms...'  latency=339.85ms  hits=2
  [healthcare_all] 'Chest X-ray findings for pneumonia...'  latency=327.97ms  hits=2
  [healthcare_all] 'Liver function test interpretation...'  latency=400.04ms  hits=2
  [healthcare_all] 'Medication allergy to ibuprofen...'  latency=335.22ms  hits=2
